In [0]:
#1 Importing Tools 
import openpyxl
import pandas as pd

from pyspark.sql import functions as F
from datetime import datetime
from openpyxl.styles import NamedStyle

In [0]:
#2 Reduce risk of a timeout by increasing limit to 30 minutes
spark.conf.set("spark.databricks.execution.timeout", "1800")

In [0]:
%skip
#3 Loading the master hierarchies table from the lake mart (via ADLS path)

# Set connection settings for prov ref data
analysisLake = "udalstdatacuratedprod.dfs.core.windows.net"
analysisContainer = "reporting"
provFile = "/unrestricted/reference/UKHD/ODS/Provider_Hierarchies_ICB"

provPath = f"abfss://{analysisContainer}@{analysisLake}{provFile}"

# Load provider reference data
df_master_hierarchies = (
    spark.read
         .option("header", "true")
         .option("recursiveFileLookup", "true")
         .parquet(provPath)
)

# Display sample
display(df_master_hierarchies.limit(10))

# Count active records (where Effective_To is NULL)
active_count = df_master_hierarchies.filter(F.col("Effective_To").isNull()).count()

print(f"Number of rows in master hierarchies (active records): {active_count}")

In [0]:
# 3a where looking at the old provider look up
from pyspark.sql import functions as F

# New file path
provPath = "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/0_Prov hierarchies.csv"

# Load provider reference data (CSV instead of parquet)
df_master_hierarchies = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(provPath)
)

# Rename STP columns to ICB columns for compatibility with downstream logic
df_master_hierarchies = (
    df_master_hierarchies
    .withColumnRenamed("STP_Code", "ICB_Code")
    .withColumnRenamed("STP_Name", "ICB_Name")
)

# Display sample
display(df_master_hierarchies.limit(10))



In [0]:
#4 loading ICB to Region mapping table
df_icb_region = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_ICB_Region_DisplayNames.csv")  

#display(df_icb_region.limit(10))
print(f"Number of rows in icb_region: {df_icb_region.count()}")

In [0]:
#5 loading list of merged providers
df_merged_providers = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_Merged_Providers.csv")

#display(df_merged_providers.limit(10))
print(f"Number of rows in merged providers: {df_merged_providers.count()}")

In [0]:
#6 creating new provider code from the provider mapping table
provider_code_mapping = df_merged_providers = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_Merged_Providers.csv")

#display(df_merged_providers.limit(10))
print(f"Number of rows in merged providers: {df_merged_providers.count()}")

In [0]:
#7 Importing the MHS details

#7a importing MHS metric list and internal ID
mhs_metric_list = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS")

#display(mhs_metric_list.limit(10))
print(f"Number of rows in mhs_metric_list: {mhs_metric_list.count()}")

#7b importing MHS allowable org codes
mhs_allowable_orgs = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS/Allowable_Org_Codes_Status.csv")

#display(mhs_allowable_orgs.limit(10))
print(f"Number of rows in mhs_allowable_orgs: {mhs_allowable_orgs.count()}")

In [0]:
#8 Loading the core monthly snapshot data
from pyspark.sql import functions as F
df_op_activity_snapshot = spark.read.option("header", "true").option("recursiveFileLookup", "true").parquet(
    "abfss://reporting@udalstdatacuratedprod.dfs.core.windows.net/restricted/patientlevel/MESH/OPA/OPA_Core_Monthly_Snapshot/Published/1"
)
display(df_op_activity_snapshot.limit(10))

# Show number of rows in the raw data
row_count = df_op_activity_snapshot.count()
print(f"Number of rows in raw data: {row_count}")

In [0]:
#9 Creating the wide table & inserting new columns for merged providers, with new merger codes and mapping to ICB and Region codes
from pyspark.sql.functions import (
    when,
    col,
    lit,
    create_map,
    coalesce,
    last_day,
    to_date,
    concat_ws,
    current_date,
    add_months,
    date_format
)
import pyspark.sql.functions as F

# 1. Define valid treatment function codes
VALID_TREATMENT_CODES = [
    '100', '101', '102', '104', '105', '106', '108', '110', '111', '115', '120', '130', '140',
    '144', '145', '301', '302', '303', '307', '320', '330', '340', '361', '400', '410', '420',
    '430', '501', '502', '560', '650'
]

# 2. Add Treatment_Function_Code_New (valid vs Other)
opa_with_tfc = df_op_activity_snapshot.withColumn(
    "Treatment_Function_Code_New",
    when(col("Treatment_Function_Code").isin(VALID_TREATMENT_CODES),
         col("Treatment_Function_Code")
    ).otherwise("Other")
)

# 3. Add Treatment_Function_Group (GS / OMFS / T&O / code)
opa_with_groups = opa_with_tfc.withColumn(
    "Treatment_Function_Group",
    when(col("Treatment_Function_Code_New").isin("100", "102", "104", "105", "106"), "GS")
    .when(col("Treatment_Function_Code_New").isin("140", "144", "145"), "OMFS")
    .when(col("Treatment_Function_Code_New").isin("110", "111", "115"), "T&O")
    .otherwise(col("Treatment_Function_Code_New"))
)

# 4. Filter dataset (years, admin category, TFC, attendance)
opa_filtered = opa_with_groups.filter(
    (col("Der_Financial_Year").isin("2023/24", "2024/25", "2025/26")) &  # Update manually if needed
    (col("Administrative_Category") == "01") &
    (col("Treatment_Function_Code") != "812") &
    (col("First_Attendance").isin("1", "2", "3", "4"))
)

# Keep ALL months up to last full month (excludes future months)
last_full_month_yyyymm = date_format(add_months(current_date(), -1), "yyyyMM")

# reinforces start month (April 2023 = 202304) 
start_month_yyyymm = F.lit("202304")  #change if needed

opa_filtered = opa_filtered.filter(
    (col("Der_Activity_Month") >= start_month_yyyymm) &
    (col("Der_Activity_Month") <= last_full_month_yyyymm)
)

# 5. Aggregate metrics by month, provider, and TFC group
opa_agg = opa_filtered.groupBy(
    "Der_Activity_Month",
    "Der_Provider_Code",
    "Treatment_Function_Group"
).agg(
    # All metrics
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "3")), 1
        ).otherwise(0)
    ).alias("All_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU"),

    F.sum(
        when(
            (col("Der_Number_Procedure") > 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_Proc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") == 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_NoProc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") > 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_Proc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") == 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_NoProc"),

    # Face-to-face
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2")), 1
        ).otherwise(0)
    ).alias("F2F_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "1"), 1
        ).otherwise(0)
    ).alias("F2F_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "2"), 1
        ).otherwise(0)
    ).alias("F2F_FU"),

    # Remote
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("3", "4")), 1
        ).otherwise(0)
    ).alias("Remote_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "3"), 1
        ).otherwise(0)
    ).alias("Remote_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "4"), 1
        ).otherwise(0)
    ).alias("Remote_FU"),

    # Did not attends (DNAs)
    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "3")), 1
        ).otherwise(0)
    ).alias("All_First_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2")), 1
        ).otherwise(0)
    ).alias("F2F_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("3", "4")), 1
        ).otherwise(0)
    ).alias("Remote_DNA"),

    # 2WW DNA
    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_2WW_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "3")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_First_2WW_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("2", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_FU_2WW_DNA"),

    # All 2WW appointments
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_2WW"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("1", "3")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_First_2WW"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("2", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_FU_2WW")
)

# 6. Add "All" TFC totals by month and provider
METRIC_COLS = [
    c for c in opa_agg.columns
    if c not in ["Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group"]
]

opa_all_tfc = opa_agg.groupBy("Der_Activity_Month", "Der_Provider_Code").agg(
    *[F.sum(col(c)).alias(c) for c in METRIC_COLS]
).withColumn("Treatment_Function_Group", lit("All"))

opa_final = opa_agg.unionByName(opa_all_tfc)

# Order by results
opa_final_ordered = opa_final.orderBy("Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group")

# 7. Normalise provider codes > Provider_Base_Code. (replicates the SQL logic)
opa_final_ordered = opa_final_ordered.withColumn(
    "Provider_Base_Code",
    when((
        col("Der_Provider_Code").substr(4, 2) == "00") & (col("Der_Provider_Code") != "NQT00"),   # e.g. ABC00
        col("Der_Provider_Code").substr(1, 3)                                                     # -> ABC
    ).otherwise(
        col("Der_Provider_Code")                                                                  # everything else is unchanged
    )
)

# 8. Build mapping from df_merged_providers (Old -> New)
provider_code_mapping_dict = {
    row["Old_Provider_Code"]: row["New_Provider_Code"]
    for row in df_merged_providers.select("Old_Provider_Code", "New_Provider_Code").distinct().collect()
}

# where k, v is present it is just short hand and means key value
mapping_list = []
for k, v in provider_code_mapping_dict.items():
    mapping_list.append(lit(k))
    mapping_list.append(lit(v))

mapping_expr = create_map(*mapping_list)

# 9. Add "Adj Org Code" based on Provider_Base_Code & mapping
opa_final_ordered_with_adj = opa_final_ordered.withColumn(
    "Adj Org Code",
    coalesce(mapping_expr.getItem(col("Provider_Base_Code")), col("Provider_Base_Code"))
)

# 10. Build Org_Code_For_Join to match hierarchy
opa_with_join_code = opa_final_ordered_with_adj.withColumn(
    "Org_Code_For_Join",
    when(
        col("Adj Org Code") == "NQT00",
        col("Adj Org Code")
    ).when(
        (F.length(col("Adj Org Code")) == 5) & col("Adj Org Code").endswith("00"),
        col("Adj Org Code").substr(1, 3)
    ).otherwise(col("Adj Org Code"))
)

# 11. Join to master hierarchies to get ICB (ICB_Code)
opa_with_icb = opa_with_join_code.join(
    df_master_hierarchies.select(
        F.col("Organisation_Code").alias("join_org_code"),
        F.col("ICB_Code").alias("ICB")
    ),
    opa_with_join_code["Org_Code_For_Join"] == F.col("join_org_code"),
    "left"
).drop("join_org_code")

# 12. Join to df_icb_region to get Region_Code
opa_with_icb_region = opa_with_icb.join(
    df_icb_region.select(
        F.col("ICB_Code").alias("join_icb"),
        F.col("Region_Code")
    ),
    opa_with_icb["ICB"] == F.col("join_icb"),
    "left"
).drop("join_icb")

# 13. Build Der_Activity_Month_Date (YYYYMM -> month end date)
opa_with_icb_region = opa_with_icb_region.withColumn(
    "Der_Activity_Month_Date",
    last_day(
        to_date(
            concat_ws(
                "-",
                col("Der_Activity_Month").substr(1, 4),
                col("Der_Activity_Month").substr(5, 2),
                lit("01")
            )
        )
    )
)

# 14. Aggregate to final level & sort
id_cols = ["Der_Activity_Month_Date", "Treatment_Function_Group", "Region_Code", "ICB", "Adj Org Code"]

metric_cols = [
    c for c in opa_with_icb_region.columns
    if c not in id_cols + [
        "Der_Activity_Month",
        "Der_Provider_Code",
        "Treatment_Function_Code_New",
        "Provider_Base_Code",
        "Org_Code_For_Join"
    ]
]

opa_final_processed = (
    opa_with_icb_region
    .groupBy(*[F.col(c) for c in id_cols])
    .agg(*[F.sum(F.col(c)).alias(c) for c in metric_cols])
    .orderBy("Der_Activity_Month_Date", "Region_Code", "ICB", "Adj Org Code", "Treatment_Function_Group")
)

display(opa_final_processed.limit(10))
print(f"opa_final_processed: {opa_final_processed.count()}")


In [0]:
#10 — Safe metric calculation (robust to missing columns) | adding all the extra metrics to the wide table
from pyspark.sql import functions as F

df = df = opa_final_processed 

def safe_add(df, new_col, expr_fn, required_cols):
    if all(c in df.columns for c in required_cols):
        return df.withColumn(new_col, expr_fn(df))
    else:
        return df.withColumn(new_col, F.lit(None))

metrics = [
    ("All_DNA_Over_All_Total", lambda d: F.when(
        (F.col("All_Total") + F.col("All_DNA")) != 0,
        (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
    ), ["All_Total", "All_DNA"]),
    ("All_DNA_Over_All_Total_IG", lambda d: F.when(
        (F.col("All_Total") + F.col("All_DNA")) != 0,
        (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
    ), ["All_Total", "All_DNA"]),
    ("All_First_DNA_Over_All_First", lambda d: F.when(
        (F.col("All_First") + F.col("All_First_DNA")) != 0,
        (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
    ), ["All_First", "All_First_DNA"]),
    ("All_First_DNA_Over_All_First_IG", lambda d: F.when(
        (F.col("All_First") + F.col("All_First_DNA")) != 0,
        (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
    ), ["All_First", "All_First_DNA"]),
    ("All_FU_DNA_Over_All_FU", lambda d: F.when(
        (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
        (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
    ), ["All_FU", "All_FU_DNA"]),
    ("All_FU_DNA_Over_All_FU_IG", lambda d: F.when(
        (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
        (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
    ), ["All_FU", "All_FU_DNA"]),
    ("All_2WW_DNA_Over_All_2WW", lambda d: F.when(
        (F.col("All_2WW") != 0) & (F.col("All_2WW").isNotNull()),
        (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100
    ), ["All_2WW_DNA", "All_2WW"]),
    ("All_FU_2WW_DNA_Over_All_FU_2WW", lambda d: F.when(
        (F.col("All_FU_2WW") != 0) & (F.col("All_FU_2WW").isNotNull()),
        (F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100
    ), ["All_FU_2WW_DNA", "All_FU_2WW"]),
    ("All_First_2WW_DNA_Over_All_First_2WW", lambda d: F.when(
        (F.col("All_First_2WW") != 0) & (F.col("All_First_2WW").isNotNull()),
        (F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100
    ), ["All_First_2WW_DNA", "All_First_2WW"]),
    ("All_FU_NoProc_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100
    ), ["All_FU_NoProc", "All_FU"]),
    ("All_FU_Proc_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100
    ), ["All_FU_Proc", "All_FU"]),
    ("All_FU_To_All_First", lambda d: F.when(
        F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))
    ), ["All_FU", "All_First"]),
    ("All_FU_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100
    ), ["All_FU", "All_Total"]),
    ("All_First_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100
    ), ["All_First", "All_Total"]),
    ("All_NoProc_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100
    ), ["All_NoProc", "All_Total"]),
    ("All_Proc_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100
    ), ["All_Proc", "All_Total"]),
    ("Remote_Total_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100
    ), ["Remote_Total", "All_Total"]),
    ("Remote_FU_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100
    ), ["Remote_FU", "All_FU"]),
    ("Remote_First_Over_All_First", lambda d: F.when(
        F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100
    ), ["Remote_First", "All_First"]),
    ("F2F_DNA_Over_F2F_Total", lambda d: F.when(
        (F.col("F2F_Total") + F.col("F2F_DNA")) != 0,
        (F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100
    ), ["F2F_Total", "F2F_DNA"]),
    ("Remote_DNA_Over_Remote_Total", lambda d: F.when(
        (F.col("Remote_Total") + F.col("Remote_DNA")) != 0,
        (F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100
    ), ["Remote_Total", "Remote_DNA"]),
]

for name, expr, req in metrics:
    df = safe_add(df, name, expr, req)

simple_copies = [
    ("All_DNA1", "All_DNA"),
    ("All_DNA2", "All_DNA"),
    ("All_First1", "All_First"),
    ("All_First2", "All_First"),
    ("All_First3", "All_First"),
    ("All_FU1", "All_FU"),
    ("All_FU2", "All_FU"),
    ("All_FU3", "All_FU"),
    ("All_FU4", "All_FU"),
    ("All_FU5", "All_FU"),
    ("All_Total1", "All_Total"),
    ("All_Total2", "All_Total"),
    ("All_Total3", "All_Total"),
    ("All_Total4", "All_Total"),
    ("All_Total5", "All_Total"),
    ("All_Total6", "All_Total"),
    ("Remote_Total1", "Remote_Total"),
    ("Remote_Total2", "Remote_Total"),
]
for newc, base in simple_copies:
    if base in df.columns:
        df = df.withColumn(newc, F.col(base))
    else:
        df = df.withColumn(newc, F.lit(None))

combos = [
    ("All_First_plus_All_First_DNA", ["All_First", "All_First_DNA"]),
    ("All_FU_plus_All_FU_DNA", ["All_FU", "All_FU_DNA"]),
    ("All_Total_plus_All_DNA", ["All_Total", "All_DNA"]),
    ("F2F_Total_plus_F2F_DNA", ["F2F_Total", "F2F_DNA"]),
    ("Remote_Total_plus_Remote_DNA", ["Remote_Total", "Remote_DNA"]),
]
for newc, cols in combos:
    if all(c in df.columns for c in cols):
        df = df.withColumn(newc, F.col(cols[0]).cast("long") + F.col(cols[1]).cast("long"))
    else:
        df = df.withColumn(newc, F.lit(None))

cols_to_drop = [c for c in ["Der_Activity_Month"] if c in df.columns]
opa_final_with_added_metrics = df.drop(*cols_to_drop)

display(opa_final_with_added_metrics.limit(10))
print(f"opa_final_with_added_metrics: {opa_final_with_added_metrics.count()} rows")

In [0]:
# 11 reshapes the wide outpatient dataset into a long (tidy) format for easier analysis
from pyspark.sql.functions import col, explode, array, struct, lit, concat_ws

# ID columns to keep
id_cols = [
    "Der_Activity_Month_Date",
    "Treatment_Function_Group",
    "Adj Org Code",
    "ICB",
    "Region_Code"
]

# Identify all metric columns
metric_cols = [c for c in opa_final_with_added_metrics.columns if c not in id_cols]

# Unpivot all metrics, casting values to string to avoid type errors
opa_long = (
    opa_final_with_added_metrics
    .select(
        *id_cols,
        explode(array(*[
            struct(
                lit(c).alias("Metric_Name"),
                col(c).cast("string").alias("Metric_Value")
            ) for c in metric_cols
        ])).alias("kv")
    )
    .select(
        *id_cols,
        col("kv.Metric_Name"),
        col("kv.Metric_Value")
    )
)

# Create the combined metric name
opa_long = opa_long.withColumn(
    "Metric_Name_Treatment_Function_Group",
    concat_ws("_", col("Metric_Name"), col("Treatment_Function_Group"))
)

# Order by date
opa_long_ordered = opa_long.orderBy("Der_Activity_Month_Date")

display(opa_long_ordered.limit(10))
#print(f"Number of rows in opa_long_ordered: {opa_long_ordered.count()}"))

In [0]:

# 12 — Aggregation for Org / ICB / Region (all sectors)
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit

# 1. Start from Org-level wide dataset from Container 10

df_org = opa_final_with_added_metrics.withColumnRenamed("Adj Org Code", "Adj_Org_Code")

# 2. (Optional) Attach sector type from df_master_hierarchies - Kept for reference / QA, not used in rollups
   
sector_lookup = (
    df_master_hierarchies
    .select(
        col("Organisation_Code").alias("ODS_Code"),
        col("ODS_Organisation_Type")
    )
    .distinct()
)

df_org = (
    df_org
    .join(
        sector_lookup,
        df_org["Adj_Org_Code"] == col("ODS_Code"),
        "left"
    )
    .drop("ODS_Code")
)

# 3. Metric columns to sum at ICB/Region

BASE_COUNT_COLS = [
    "All_Total","All_First","All_FU","All_Proc","All_NoProc",
    "All_FU_Proc","All_FU_NoProc",
    "F2F_Total","F2F_First","F2F_FU","F2F_DNA",
    "Remote_Total","Remote_First","Remote_FU","Remote_DNA",
    "All_DNA","All_First_DNA","All_FU_DNA",
    "All_2WW","All_First_2WW","All_FU_2WW",
    "All_2WW_DNA","All_First_2WW_DNA","All_FU_2WW_DNA"
]

# Small safety feature : only sum columns that actually exist
count_cols = [c for c in BASE_COUNT_COLS if c in df_org.columns]

# 4. Function to add rate metrics

def add_rate_metrics(df):
    return (
        df
        # DNA metrics
        .withColumn("All_DNA_Over_All_Total", F.when(
            (F.col("All_Total") + F.col("All_DNA")) != 0,
            (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
        ))
        .withColumn("All_DNA_Over_All_Total_IG", F.col("All_DNA_Over_All_Total"))
        .withColumn("All_First_DNA_Over_All_First", F.when(
            (F.col("All_First") + F.col("All_First_DNA")) != 0,
            (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
        ))
        .withColumn("All_First_DNA_Over_All_First_IG", F.col("All_First_DNA_Over_All_First"))
        .withColumn("All_FU_DNA_Over_All_FU", F.when(
            (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
            (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
        ))
        .withColumn("All_FU_DNA_Over_All_FU_IG", F.col("All_FU_DNA_Over_All_FU"))
        # FU metrics
        .withColumn("All_FU_NoProc_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100
        ))
        .withColumn("All_FU_Proc_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100
        ))
        .withColumn("All_FU_To_All_First", F.when(
            F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))
        ))
        # 2WW
        .withColumn("All_2WW_DNA_Over_All_2WW", F.when(
            F.col("All_2WW") != 0, (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100
        ))
        .withColumn("All_First_2WW_DNA_Over_All_First_2WW", F.when(
            F.col("All_First_2WW") != 0, (F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100
        ))
        .withColumn("All_FU_2WW_DNA_Over_All_FU_2WW", F.when(
            F.col("All_FU_2WW") != 0, (F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100
        ))
        # Mix shares
        .withColumn("All_FU_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100
        ))
        .withColumn("All_First_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100
        ))
        .withColumn("All_NoProc_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100
        ))
        .withColumn("All_Proc_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100
        ))
        .withColumn("Remote_Total_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100
        ))
        .withColumn("Remote_FU_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100
        ))
        .withColumn("Remote_First_Over_All_First", F.when(
            F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100
        ))
        .withColumn("Remote_DNA_Over_Remote_Total", F.when(
            (F.col("Remote_Total") + F.col("Remote_DNA")) != 0,
            (F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100
        ))
        .withColumn("F2F_DNA_Over_F2F_Total", F.when(
            (F.col("F2F_Total") + F.col("F2F_DNA")) != 0,
            (F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100
        ))
    )

# 5. Aggregate to ICB and Region (all orgs included)

df_icb = (
    df_org
    .groupBy("Der_Activity_Month_Date", "ICB", "Treatment_Function_Group")
    .agg(*[F.sum(col(c)).alias(c) for c in count_cols])
)
df_icb = add_rate_metrics(df_icb).withColumn("Level", lit("ICB"))

df_region = (
    df_org
    .groupBy("Der_Activity_Month_Date", "Region_Code", "Treatment_Function_Group")
    .agg(*[F.sum(col(c)).alias(c) for c in count_cols])
)
df_region = add_rate_metrics(df_region).withColumn("Level", lit("Region"))

# 6. Org-level rows

df_org = df_org.withColumn("Level", lit("Org"))

# 7. Combine all levels

final_output = (
    df_org.unionByName(df_icb, allowMissingColumns=True)
          .unionByName(df_region, allowMissingColumns=True)
)

# 8. Final code column

final_output = final_output.withColumn(
    "Adj_Org_Code_Final",
    when(col("Level") == "Org", col("Adj_Org_Code"))
    .when(col("Level") == "ICB", col("ICB"))
    .when(col("Level") == "Region", col("Region_Code"))
)

# 9. Simple - copy and combo columns

copy_map = {
    "All_DNA1": "All_DNA",
    "All_DNA2": "All_DNA",
    "All_FU1": "All_FU",
    "All_FU2": "All_FU",
    "All_FU3": "All_FU",
    "All_FU4": "All_FU",
    "All_FU5": "All_FU",
    "All_First1": "All_First",
    "All_First2": "All_First",
    "All_First3": "All_First",
    "All_Total1": "All_Total",
    "All_Total2": "All_Total",
    "All_Total3": "All_Total",
    "All_Total4": "All_Total",
    "All_Total5": "All_Total",
    "All_Total6": "All_Total",
    "Remote_Total1": "Remote_Total",
    "Remote_Total2": "Remote_Total"
}

for newc, base in copy_map.items():
    if base in final_output.columns:
        final_output = final_output.withColumn(newc, col(base))

combo_pairs = {
    "All_First_plus_All_First_DNA": ("All_First", "All_First_DNA"),
    "All_FU_plus_All_FU_DNA": ("All_FU", "All_FU_DNA"),
    "All_Total_plus_All_DNA": ("All_Total", "All_DNA"),
    "F2F_Total_plus_F2F_DNA": ("F2F_Total", "F2F_DNA"),
    "Remote_Total_plus_Remote_DNA": ("Remote_Total", "Remote_DNA")
}

for newc, (a, b) in combo_pairs.items():
    if a in final_output.columns and b in final_output.columns:
        final_output = final_output.withColumn(newc, col(a).cast("long") + col(b).cast("long"))

# Final preview
display(final_output.limit(20))
print("Container 12 complete —", final_output.count(), "rows")

In [0]:
# 13 – long/skinny OPRT format (Level preserved)
from pyspark.sql.functions import col, lit, explode, array, struct, when
import pyspark.sql.functions as F

# Use the final_output table from container 12 (has Level + Adj_Org_Code_Final)
df_wide = final_output

# Step 1: Drop unnecessary columns
cols_to_drop = [
    "Adj_Org_Code",
    "All_DNA", "All_First", "All_FU",
    "All_Total", "F2F_Total", "Remote_Total"
]
df_wide = df_wide.drop(*[c for c in cols_to_drop if c in df_wide.columns])

# Step 2: Define identifier columns (KEEP Level)
id_cols = [
    "Der_Activity_Month_Date",
    "Region_Code",
    "ICB",
    "Adj_Org_Code_Final",
    "Treatment_Function_Group",
    "Level",
]

# Step 3: Identify metric columns (exclude identifiers)
metric_cols = [c for c in df_wide.columns if c not in id_cols]

# Helper: safe numeric cast under ANSI mode, If value looks like a number → cast to double, Else → NULL

def safe_numeric(colname: str):
    return (
        when(
            col(colname).rlike(r'^[-+]?[0-9]*\.?[0-9]+$'),
            col(colname).cast("double")
        )
        .otherwise(F.lit(None).cast("double"))
        .alias("Metric_Value")
    )

# Step 4: Melt into long/skinny format
opa_oprt_long = (
    df_wide.select(
        *id_cols,
        explode(array(*[
            struct(
                lit(c).alias("OPRT_Metric_Name"),
                safe_numeric(c)
            )
            for c in metric_cols
        ])).alias("kv")
    )
    .select(
        *id_cols,
        col("kv.OPRT_Metric_Name"),
        col("kv.Metric_Value")
    )
)

# Drop null metric rows (zeros are kept)
opa_oprt_long = opa_oprt_long.filter(col("Metric_Value").isNotNull())

display(opa_oprt_long.limit(10))
print(f"Container 13 complete — {opa_oprt_long.count()} rows, {len(opa_oprt_long.columns)} columns")

# Ensures Container 14 uses the cleaned long-format table
final_output = opa_oprt_long

# Write opa_oprt_long to single CSV file

opa_oprt_long.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects/opa_oprt_long.csv")

print("opa_oprt_long CSV export completed.")


In [0]:
# 14 Prepare Preview Outpatient Metrics and Map Internal IDs
from pyspark.sql.functions import col, concat_ws, lower, regexp_replace, trim
from pyspark.sql.types import StringType, DoubleType, DateType
from pyspark.storagelevel import StorageLevel

# Start from Cell 13 output
df_outpat_metrics = final_output.filter(col("Treatment_Function_Group") != "Other")

# Build combined metric name for joining to ID list
df_outpat_metrics = df_outpat_metrics.withColumn(
    "OPRT_Metric_Name_TFC",
    concat_ws("_", col("OPRT_Metric_Name"), col("Treatment_Function_Group"))
)

# Helper to standardise metric join keys
def normalise_metric_key(col_expr):
    return regexp_replace(
        regexp_replace(
            regexp_replace(
                lower(regexp_replace(trim(col_expr), r"\s+", "_")),
                r"[^a-z0-9_]", ""
            ),
            r"_+", "_"
        ),
        r"^_+|_+$", ""
    )

# Normalise join key on long dataset
df_outpat_metrics = df_outpat_metrics.withColumn(
    "join_metric",
    normalise_metric_key(col("OPRT_Metric_Name_TFC"))
)

# Identify the OPRT TFC metric column in the reference list
metric_list_cols = [c for c in mhs_metric_list.columns if "OPRT" in c and "TFC" in c]
if len(metric_list_cols) == 0:
    raise ValueError(
        "Could not find OPRT TFC metric column name in mhs_metric_list. Review mhs_metric_list columns."
    )
metric_list_col = metric_list_cols[0]

# Prepare cleaned metric lookup
mhs_metric_list_clean = mhs_metric_list.withColumn(
    "join_metric",
    normalise_metric_key(col(metric_list_col))
)

select_cols = ["join_metric", "InternalID"]
if "Description" in mhs_metric_list_clean.columns:
    select_cols.append("Description")
else:
    print("WARNING: mhs_metric_list does not contain a column named 'Description'.")

mhs_metric_list_clean_sel = mhs_metric_list_clean.select(*select_cols).distinct()

# Optional debug join to inspect unmatched metrics
df_outpat_metrics_left = df_outpat_metrics.join(
    mhs_metric_list_clean_sel,
    on="join_metric",
    how="left"
)

unmatched = (
    df_outpat_metrics_left
    .filter(col("InternalID").isNull())
    .select("OPRT_Metric_Name_TFC")
    .distinct()
)

# Final production join (drop unmapped rows)
df_outpat_metrics = (
    df_outpat_metrics
    .join(mhs_metric_list_clean_sel, on="join_metric", how="inner")
    .drop("join_metric")
)

# Filter allowable org codes if reference provided
if "Org_Code" in mhs_allowable_orgs.columns:
    df_outpat_metrics = df_outpat_metrics.join(
        mhs_allowable_orgs.select(col("Org_Code").alias("allowable_code")),
        df_outpat_metrics["Adj_Org_Code_Final"] == col("allowable_code"),
        "inner"
    ).drop("allowable_code")

# Enforce clean data types
df_outpat_metrics = (
    df_outpat_metrics
    .withColumn("Der_Activity_Month_Date", col("Der_Activity_Month_Date").cast(DateType()))
    .withColumn("Adj_Org_Code_Final", col("Adj_Org_Code_Final").cast(StringType()))
    .withColumn("Level", col("Level").cast(StringType()))
    .withColumn("Treatment_Function_Group", col("Treatment_Function_Group").cast(StringType()))
    .withColumn("OPRT_Metric_Name", col("OPRT_Metric_Name").cast(StringType()))
    .withColumn("OPRT_Metric_Name_TFC", col("OPRT_Metric_Name_TFC").cast(StringType()))
    .withColumn("Metric_Value", col("Metric_Value").cast(DoubleType()))
)

# Final output column names
df_outpat_metrics = (
    df_outpat_metrics
    .withColumnRenamed("Der_Activity_Month_Date", "reportingDate")
    .withColumnRenamed("Adj_Org_Code_Final", "providerCode")
    .withColumnRenamed("InternalID", "metricID")
    .withColumnRenamed("Metric_Value", "value")
)

# Keep only required columns
df_outpat_metrics = df_outpat_metrics.select("reportingDate", "providerCode", "metricID", "value")

# Persist once for reuse in later cells
df_outpat_metrics = df_outpat_metrics.persist(StorageLevel.MEMORY_AND_DISK)
_ = df_outpat_metrics.count()

display(df_outpat_metrics.limit(20))

# Write df_outpat_metrics to single CSV file

#df_outpat_metrics.coalesce(1).write \
    #.format("csv") \
    #.mode("overwrite") \
    #.option("header", "true") \
    #.save("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects/df_outpat_metrics.csv")


In [0]:
# 15 Extract Allowable OZ Metric Lists and Pull Matching Production Data
import openpyxl
import sys

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES

# Excel source
xlsx_path = "/Workspace/Users/steven.evans4@udal.nhs.uk/MHS-ingestion-for-Outpatients/op_ref.xlsx"
sheet_name = "Lists and df"

groups = [
    ("df_fofu", 4),
    ("df_miss_all", 9),
    ("df_miss_fst", 14),
    ("df_miss_f2f", 19),
    ("df_miss_2ww", 24),
    ("df_fu_nop", 29),
    ("df_remote", 34),
    ("df_all", 39),
]

def extract_metric_id_lists(xlsx_path: str, sheet_name: str, groups, start_row: int = 6):
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)

    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {wb.sheetnames}")

    ws = wb[sheet_name]
    out = {}

    for name, metric_col in groups:
        ids = []

        for r in range(start_row, ws.max_row + 1):
            v = ws.cell(r, metric_col).value

            if v is None or str(v).strip() == "":
                if ids:
                    lookahead_end = True
                    for rr in range(r, min(r + 6, ws.max_row + 1)):
                        vv = ws.cell(rr, metric_col).value
                        if vv is not None and str(vv).strip() != "":
                            lookahead_end = False
                            break
                    if lookahead_end:
                        break
                continue

            ids.append(str(v).strip())

        seen = set()
        ids = [x for x in ids if not (x in seen or seen.add(x))]

        if len(ids) == 0:
            print(f"WARNING: '{name}' list extracted 0 Metric IDs.")

        out[name] = ids

    return out

metric_lists = extract_metric_id_lists(xlsx_path, sheet_name, groups)

# Combine all OZ codes from the extracted lists
all_metric_ids = []
seen = set()

for ids in metric_lists.values():
    for x in ids:
        if x.startswith("OZ") and x not in seen:
            seen.add(x)
            all_metric_ids.append(x)

print("Number of OZ codes:", len(all_metric_ids))
print("Example codes:", all_metric_ids[:10])

if not all_metric_ids:
    raise ValueError("No OZ metric IDs found in the workbook lists.")

# Build SQL IN clause
metric_id_sql = ", ".join("'" + x.replace("'", "''") + "'" for x in all_metric_ids)

# Connect to SQL Server
mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')

# Query Production data
query = f"""
SELECT
       CAST(mv.[reportingDate] AS DATE) AS reportingDate
      ,CAST(mv.[value] AS FLOAT) AS value
      ,m.[InternalID] AS metricID
      ,p.[Code] AS providerCode
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
LEFT JOIN [dbo].[Provider] p
    ON p.ID = mv.ProviderID
WHERE mv.[reportingDate] > '2023-03-31'
  AND m.[InternalID] IN ({metric_id_sql})
"""

prod_df = mhs_backend_conn.read_from_db(query)

display(prod_df)

prod_row_count = prod_df.count()
display(spark.createDataFrame([(prod_row_count,)], ["row_count"]))

In [0]:
# 16 Pull matching production data from preview metric scope and build reconciliation outputs for insert notebook
# Business rules:
# - comparison window starts from April 2024 onwards
# - preview is capped to a hard-coded maximum reporting date
# - latest preview month within that cap = Adding
# - historical matched rows with material value change = Revising
# - historical preview-only rows = QA only, not included in final insert
# - prod-only historical rows = Deleting candidates for QA/reference
# - this notebook only inserts Adding + Revising

import sys
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, StringType, DoubleType
from pyspark.storagelevel import StorageLevel

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES

spark.catalog.clearCache()

# -----------------------------
# Parameters
# -----------------------------
RECON_CUTOFF_DATE = "2024-04-01"  # *** CAN BE CHANGED TO EARLIER / LATER *** #
MAX_PREVIEW_DATE = "2026-02-28"   # *** NEEDS CHANGING EVERY MONTH *** #
REVISION_TOLERANCE = 0.1

# -----------------------------
# Restrict preview to agreed maximum reporting date
# This prevents incomplete / unpublished months being included
# -----------------------------
preview_metric_scope_df = (
    df_outpat_metrics
    .withColumn("reportingDate", F.col("reportingDate").cast(DateType()))
    .filter(F.col("reportingDate") <= F.to_date(F.lit(MAX_PREVIEW_DATE)))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = preview_metric_scope_df.count()

# -----------------------------
# Build production metric scope directly from preview
# This replaces the old Excel group logic
# -----------------------------
metric_id_rows = (
    preview_metric_scope_df
    .select("metricID")
    .filter(F.col("metricID").isNotNull())
    .distinct()
    .collect()
)

all_metric_ids = sorted({str(row["metricID"]).strip() for row in metric_id_rows if row["metricID"] is not None})

print("Number of metric IDs in preview scope:", len(all_metric_ids))
print("Example metric IDs:", all_metric_ids[:10])

if not all_metric_ids:
    raise ValueError("No metric IDs found in preview scope after applying MAX_PREVIEW_DATE filter.")

metric_id_sql = ", ".join("'" + x.replace("'", "''") + "'" for x in all_metric_ids)

# -----------------------------
# Connect to SQL Server and pull matching production rows
# -----------------------------
mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')

query = f"""
SELECT
       CAST(mv.[reportingDate] AS DATE) AS reportingDate
      ,CAST(mv.[value] AS FLOAT) AS value
      ,m.[InternalID] AS metricID
      ,p.[Code] AS providerCode
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
LEFT JOIN [dbo].[Provider] p
    ON p.ID = mv.ProviderID
WHERE mv.[reportingDate] > '2023-03-31'
  AND m.[InternalID] IN ({metric_id_sql})
"""

prod_df = mhs_backend_conn.read_from_db(query)

display(prod_df.limit(20))

prod_row_count = prod_df.count()
display(spark.createDataFrame([(prod_row_count,)], ["row_count"]))

# -----------------------------
# Standardise production
# -----------------------------
prod_compare = (
    prod_df
    .withColumn("reportingDate", F.col("reportingDate").cast(DateType()))
    .withColumn("metricID", F.trim(F.col("metricID")).cast(StringType()))
    .withColumn("providerCode", F.trim(F.col("providerCode")).cast(StringType()))
    .withColumn("value", F.col("value").cast(DoubleType()))
    .select("reportingDate", "metricID", "providerCode", "value")
)

# -----------------------------
# Standardise preview
# -----------------------------
preview_compare = (
    preview_metric_scope_df
    .withColumn("reportingDate", F.col("reportingDate").cast(DateType()))
    .withColumn("metricID", F.trim(F.col("metricID")).cast(StringType()))
    .withColumn("providerCode", F.trim(F.col("providerCode")).cast(StringType()))
    .withColumn("value", F.col("value").cast(DoubleType()))
    .select("reportingDate", "metricID", "providerCode", "value")
)

# -----------------------------
# Restrict comparison window
# -----------------------------
cutoff_date = F.to_date(F.lit(RECON_CUTOFF_DATE))

prod_compare = prod_compare.filter(F.col("reportingDate") >= cutoff_date)
preview_compare = preview_compare.filter(F.col("reportingDate") >= cutoff_date)

# -----------------------------
# Aggregate to ingestion grain (rounded to 2dp to align with Alteryx)
# -----------------------------
prod_agg = (
    prod_compare
    .groupBy("reportingDate", "metricID", "providerCode")
    .agg(F.round(F.sum("value"), 2).alias("prod_value"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

preview_agg = (
    preview_compare
    .groupBy("reportingDate", "metricID", "providerCode")
    .agg(F.round(F.sum("value"), 2).alias("preview_value"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = prod_agg.count()
_ = preview_agg.count()

# -----------------------------
# Identify date sets
# -----------------------------
prod_dates_df = prod_agg.select("reportingDate").distinct()
preview_dates_df = preview_agg.select("reportingDate").distinct()

common_dates_df = preview_dates_df.join(prod_dates_df, on="reportingDate", how="inner")
preview_only_dates_df = preview_dates_df.join(prod_dates_df, on="reportingDate", how="left_anti")
prod_only_dates_df = prod_dates_df.join(preview_dates_df, on="reportingDate", how="left_anti")

latest_preview_date = (
    preview_dates_df
    .agg(F.max("reportingDate").alias("latest_preview_date"))
    .first()["latest_preview_date"]
)

if latest_preview_date is None:
    raise ValueError(
        f"No preview reportingDate found on or after {RECON_CUTOFF_DATE} "
        f"and on or before {MAX_PREVIEW_DATE}."
    )

print(f"latest_preview_date = {latest_preview_date}")
print(f"max_preview_date = {MAX_PREVIEW_DATE}")

# -----------------------------
# Adding bucket
# Latest preview month full load
# -----------------------------
df_adding = (
    preview_agg
    .filter(F.col("reportingDate") == F.lit(latest_preview_date))
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        F.col("preview_value").alias("value")
    )
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_adding.count()

# -----------------------------
# Historical common-month window
# Excludes latest preview month because that month is handled as Adding
# -----------------------------
common_hist_dates_df = (
    common_dates_df
    .filter(F.col("reportingDate") < F.lit(latest_preview_date))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = common_hist_dates_df.count()

prod_hist_df = (
    prod_agg
    .join(common_hist_dates_df, on="reportingDate", how="inner")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

preview_hist_df = (
    preview_agg
    .join(common_hist_dates_df, on="reportingDate", how="inner")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = prod_hist_df.count()
_ = preview_hist_df.count()

prod_hist_keys_df = (
    prod_hist_df
    .select("reportingDate", "metricID", "providerCode")
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

preview_hist_keys_df = (
    preview_hist_df
    .select("reportingDate", "metricID", "providerCode")
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = prod_hist_keys_df.count()
_ = preview_hist_keys_df.count()

# -----------------------------
# Revising bucket
# Matched historical keys where absolute value difference is material
# Also retains preview/prod/difference for QA detail review
# -----------------------------
matched_value_compare_hist_df = (
    preview_hist_df
    .join(
        prod_hist_df,
        on=["reportingDate", "metricID", "providerCode"],
        how="inner"
    )
    .withColumn("difference_value", F.round(F.col("preview_value") - F.col("prod_value"), 2))
    .withColumn("abs_difference_value", F.abs(F.col("difference_value")))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = matched_value_compare_hist_df.count()

df_revising_detail = (
    matched_value_compare_hist_df
    .filter(F.col("abs_difference_value") > F.lit(REVISION_TOLERANCE))
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        "preview_value",
        "prod_value",
        "difference_value"
    )
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_revising_detail.count()

df_revising = (
    df_revising_detail
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        F.col("preview_value").alias("value")
    )
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_revising.count()

# -----------------------------
# Historical preview-only rows
# QA only, not included in final insert
# -----------------------------
preview_only_hist_keys_df = (
    preview_hist_keys_df
    .join(
        prod_hist_keys_df,
        on=["reportingDate", "metricID", "providerCode"],
        how="left_anti"
    )
)

df_preview_only_historical = (
    preview_only_hist_keys_df
    .join(
        preview_hist_df.select(
            "reportingDate", "metricID", "providerCode", "preview_value"
        ),
        on=["reportingDate", "metricID", "providerCode"],
        how="inner"
    )
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        F.col("preview_value").alias("value")
    )
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_preview_only_historical.count()

# -----------------------------
# Deleting bucket
# QA/reference only in this notebook
# Scoped to metric/provider pairs seen in preview history
# -----------------------------
preview_metric_provider_scope_df = (
    preview_hist_df
    .select("metricID", "providerCode")
    .distinct()
)

prod_hist_keys_scoped_df = (
    prod_hist_keys_df
    .join(
        preview_metric_provider_scope_df,
        on=["metricID", "providerCode"],
        how="inner"
    )
)

df_deleting = (
    prod_hist_keys_scoped_df
    .join(
        preview_hist_keys_df,
        on=["reportingDate", "metricID", "providerCode"],
        how="left_anti"
    )
    .withColumn("value", F.lit(None).cast(DoubleType()))
    .select("reportingDate", "providerCode", "metricID", "value")
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_deleting.count()

# -----------------------------
# Final insert output
# This notebook inserts only Adding + Revising
# -----------------------------
final_insert_df = (
    df_adding
    .unionByName(df_revising)
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = final_insert_df.count()

# -----------------------------
# Summary counts
# -----------------------------
counts_data = [
    ("prod_df", prod_df.count()),
    ("preview_metric_scope_df", preview_metric_scope_df.count()),
    ("prod_agg", prod_agg.count()),
    ("preview_agg", preview_agg.count()),
    ("common_dates_df", common_dates_df.count()),
    ("preview_only_dates_df", preview_only_dates_df.count()),
    ("prod_only_dates_df", prod_only_dates_df.count()),
    ("common_hist_dates_df", common_hist_dates_df.count()),
    ("df_adding", df_adding.count()),
    ("df_revising", df_revising.count()),
    ("df_preview_only_historical", df_preview_only_historical.count()),
    ("df_deleting", df_deleting.count()),
    ("final_insert_df", final_insert_df.count())
]

counts_df = spark.createDataFrame(counts_data, ["dataset", "row_count"])
display(counts_df)

# -----------------------------
# Final reconciliation window QA check
# -----------------------------
recon_min_date = (
    preview_compare
    .agg(F.min("reportingDate").alias("min_date"))
    .first()["min_date"]
)

recon_max_date = (
    preview_compare
    .agg(F.max("reportingDate").alias("max_date"))
    .first()["max_date"]
)

prod_min_date = (
    prod_compare
    .agg(F.min("reportingDate").alias("min_date"))
    .first()["min_date"]
)

prod_max_date = (
    prod_compare
    .agg(F.max("reportingDate").alias("max_date"))
    .first()["max_date"]
)

print("----- RECONCILIATION WINDOW CHECK -----")
print(f"Configured cutoff date        : {RECON_CUTOFF_DATE}")
print(f"Configured max preview date   : {MAX_PREVIEW_DATE}")
print("")
print(f"Preview data range used       : {recon_min_date} → {recon_max_date}")
print(f"Production data range used    : {prod_min_date} → {prod_max_date}")
print(f"Latest preview date (Adding)  : {latest_preview_date}")
print("--------------------------------------")

In [0]:
# 17 QA summary for insert notebook
from pyspark.sql import functions as F

print(f"latest_preview_date: {latest_preview_date}")
print(f"max_preview_date: {MAX_PREVIEW_DATE}")
print(f"reconciliation_cutoff_date: {RECON_CUTOFF_DATE}")
print(f"revision_tolerance: {REVISION_TOLERANCE}")

print(f"Adding rows: {df_adding.count()}")
print(f"Revising rows: {df_revising.count()}")
print(f"Historical preview-only rows (QA only): {df_preview_only_historical.count()}")
print(f"Deleting rows (QA/reference only): {df_deleting.count()}")
print(f"Final insert row count (Adding + Revising): {final_insert_df.count()}")

adding_by_date_df = (
    df_adding
    .groupBy("reportingDate")
    .agg(F.count("*").alias("row_count"))
    .orderBy("reportingDate")
)

revising_by_date_df = (
    df_revising
    .groupBy("reportingDate")
    .agg(F.count("*").alias("row_count"))
    .orderBy("reportingDate")
)

deleting_by_date_df = (
    df_deleting
    .groupBy("reportingDate")
    .agg(F.count("*").alias("row_count"))
    .orderBy("reportingDate")
)

preview_only_historical_by_date_df = (
    df_preview_only_historical
    .groupBy("reportingDate")
    .agg(F.count("*").alias("row_count"))
    .orderBy("reportingDate")
)

revising_detail_sample_df = (
    df_revising_detail
    .orderBy("reportingDate", "providerCode", "metricID")
)

display(adding_by_date_df)
display(revising_by_date_df)
display(deleting_by_date_df)
display(preview_only_historical_by_date_df)

# First 20 revising rows showing preview value, prod value and difference
display(revising_detail_sample_df.limit(20))

In [0]:
#%skip
# 18 Optional QA CSV exports
# CSV outputs for direct comparison with Alteryx

base_path = "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects"

df_adding.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_adding.csv")

df_revising.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_revising.csv")

df_revising_detail.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_revising_detail.csv")

df_deleting.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_deleting.csv")

print("QA CSV exports for Adding, Revising, Revising Detail and Deleting completed.")

In [0]:
%skip
# 19 Prepare final insert dataset for upload
# This notebook now uploads final_insert_df directly with no grouped metric splits

from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

final_insert_upload_df = (
    final_insert_df
    .select("reportingDate", "providerCode", "metricID", "value")
    .dropDuplicates()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = final_insert_upload_df.count()

display(final_insert_upload_df.limit(20))
print(f"final_insert_upload_df row count: {final_insert_upload_df.count()}")


In [0]:

%skip
# 20 Upload final insert dataset to MHS
# Note:
# - this notebook uploads INSERT rows only
# - final_insert_df = Adding + Revising
# - delete actions are handled in a separate notebook

import sys
from pyspark.sql import functions as F

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_import import MHS_IngestionHub
from MHS_Helper.mhs_db_config import INSERT

RUN_INSERT = False  # Change to True when approved for live execution

def upload_to_mhs(df_to_upload, description_text, skip_existing_check=True):
    upload_df = (
        df_to_upload
        .withColumn("reportingDate", F.to_date("reportingDate"))
        .select("reportingDate", "providerCode", "metricID", "value")
        .dropDuplicates()
    )

    MHS_IngestionHub.upload_many(
        mhs_df=upload_df,
        description=description_text,
        loaded_by="steven.evans4@nhs.net",
        mhs_mode=INSERT,
        skip_existing_data_check=skip_existing_check
    )

def main():
    upload_df = (
        final_insert_upload_df
        .select("reportingDate", "providerCode", "metricID", "value")
        .dropDuplicates()
    )

    row_count = upload_df.count()

    if row_count == 0:
        print("No rows found in final_insert_upload_df - skipping upload.")
        return

    max_date = (
        upload_df
        .agg(F.max("reportingDate").alias("max_date"))
        .collect()[0]["max_date"]
    )

    try:
        max_date_str = max_date.strftime("%Y-%m-%d")
    except Exception:
        max_date_str = str(max_date)

    description_text = f"df_outpat_metrics_insert_{max_date_str}"

    print(f"Row count: {row_count}")
    print(f"Description: {description_text}")

    display(upload_df.orderBy("reportingDate", "providerCode", "metricID").limit(20))

    if RUN_INSERT:
        print("Uploading final insert dataset...")
        upload_to_mhs(upload_df, description_text, skip_existing_check=True)
        print("Upload completed successfully.\n")
    else:
        print("RUN_INSERT = False, dry run only. No upload executed.\n")

if __name__ == "__main__":
    main()

In [0]:
%skip
# 21 Old one-off ingestion cell retired  | Original ingestion code but not in use
# This cell has been superseded by cells 17 and 18.
# Do not use.
# MHS Ingestion - original script
import sys
from pyspark.sql import functions as F

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')
from MHS_Helper.mhs_import import MHS_IngestionHub
from MHS_Helper.mhs_db_config import INSERT, DELETE


def main():

    # Only ingest metricID == 'OZ0001'
    df_filtered = df_outpat_metrics.filter(F.col("metricID") == "OZ0001")

    row_count = df_filtered.count()

    if row_count == 0:
        print("No rows found for metricID = 'OZ0001' — skipping upload.")
        return

    max_date = (
        df_filtered
        .agg(F.max("reportingDate").alias("max_date"))
        .collect()[0]["max_date"]
    )

    desc = f"df_outpat_metrics_oz0001_{max_date}"

    InjectionHub(df_filtered, desc, True)

    print(f"df_outpat_metrics (metricID = OZ0001) sent to InjectionHub — {row_count} rows")


def InjectionHub(dfih, desc, tf):

    sdf = dfih.withColumn("reportingDate", F.to_date("reportingDate"))

    display(sdf.sample(False, 0.01))
    display(sdf.dtypes)

    MHS_IngestionHub.upload_many(
        mhs_df=sdf,
        description=desc,
        loaded_by="katie.conners@nhs.net",
        mhs_mode=INSERT,
        skip_existing_data_check=tf
    )


if __name__ == '__main__':
    main()